## Organized and Finalized Training Script ##

In [31]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets
import os
import kagglehub

In [32]:
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')

print("Path to competition files:", path)

Path to competition files: C:\Users\Admin\.cache\kagglehub\competitions\house-prices-advanced-regression-techniques


### Load Data ###

In [33]:
df = pd.read_csv(os.path.join(path, 'train.csv'))

gr_liv_area = df["GrLivArea"]
garage_area = df["GarageArea"]
year_built = df["YearBuilt"]
year_remodelled = df["YearRemodAdd"]
year_sold = df["YrSold"]
overall_qual = df["OverallQual"]
overall_cond = df["OverallCond"]
neighborhood = df["Neighborhood"]
y_df = df["SalePrice"]

### Build Features ###

In [34]:
remodelled_age = year_sold - year_remodelled
built_age = year_sold - year_built
# YearRemodAdd seems to be floored at 1950 so flag those rows
remod_floor_flag = np.zeros(remodelled_age.size)
remod_floor_flag[year_remodelled == 1950] = 1

In [35]:
# convert to one hot before splitting training and validation to avoid missing any neighborhoods in either set
neighborhood_names = np.unique(neighborhood)
# Dropped NAmes (baseline neighborhood)
neighborhood_names = neighborhood_names[neighborhood_names != "NAmes"]
neighborhood_one_hot = {}
for name in neighborhood_names:
    neighborhood_one_hot[name] = (neighborhood == name).astype(float)

x_arr = np.column_stack((gr_liv_area, garage_area, overall_qual, built_age, remodelled_age, remod_floor_flag))
for n in neighborhood_one_hot:
    x_arr = np.column_stack((x_arr, neighborhood_one_hot[n]))

In [36]:
# turn y into log_y so that errors are converted from absolute to relative
log_y_arr = np.log(y_df).to_numpy()

### Split Training and Validation ###

In [ ]:
rng = np.random.default_rng(seed=912)
shuffled_indices = rng.permutation(x_arr.shape[0])

validation_cutoff = x_arr.shape[0] - int(x_arr.shape[0] / 5)
train_indices = shuffled_indices[0:validation_cutoff]
valid_indices = shuffled_indices[validation_cutoff:]

x_train = x_arr[train_indices]
x_valid = x_arr[valid_indices]

log_y_train = log_y_arr[train_indices]
log_y_valid = log_y_arr[valid_indices]

(1168, 30)
(292, 30)
(1168,)
(292,)
